In [1]:
import logging
from exp.run import ExperimentRun, SummarySectionName
from exp.config import TransformerExperiments, CNNExperiments
logging.basicConfig(level=logging.ERROR)

In [2]:
config = TransformerExperiments()
# config = CNNExperiments()
exp = ExperimentRun(config=config)

In [3]:
import torch
print(f"PyTorch version: {torch.__version__}")
print(f"PyTorch built with CUDA version: {torch.version.cuda}")
print(f"CUDA version: {torch.cuda.is_available()}")
print(f"CUDA device count: {torch.cuda.device_count()}")

PyTorch version: 2.6.0+cu124
PyTorch built with CUDA version: 12.4
CUDA version: True
CUDA device count: 2


In [4]:
models = [
    "EleutherAI/gpt-neo-125M",
    "facebook/opt-125m",
    "facebook/opt-350m",
    "cerebras/Cerebras-GPT-111M",
    "microsoft/deberta-base",
    "T5-small",
    "t5-base",
    "distilbert/distilgpt2",
    "openai-community/gpt2",
]
model = models[0]
batch = 5
optimizer = "AdamW"
gpu_id = 1
task_id = None
in_docker = True

In [5]:
# for batch in range(10, 30, 10):
#     exp.add_task(
#         model_name=model,
#         batch_size=batch,
#         optimizer=optimizer,
#         gpu_id=gpu_id,
#         task_id=task_id,
#     )
exp.add_task(
    model_name=model,
    batch_size=batch,
    optimizer=optimizer,
    gpu_id=gpu_id,
    task_id=task_id,
)



## Measure Ground Truth for Each job

In [6]:
exp.run_group_truth(in_docker=in_docker)

100%|██████████| 1/1 [00:00<00:00, 69.07it/s]


=============== Start massively run for GPU train ======================


100%|██████████| 1/1 [00:23<00:00, 23.45s/it]
0it [00:00, ?it/s]


## Estimate Max GPU Memory by Solution Described in the Paper

In [ ]:
exp.run_estimation(estimators=[SummarySectionName.solution], in_docker=False)

  0%|          | 0/1 [00:00<?, ?it/s]/home/glaswegian/miniconda3/envs/xmem-2.6/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


EleutherAI/gpt-neo-125M estimated by solution


[W421 23:32:34.127553078 CPUAllocator.cpp:245] Memory block of unknown size was allocated before the profiling started, profiler results will not include the deallocation event


## Estimate Max GPU Memory by DNNmem

In [8]:
exp.run_estimation(estimators=[SummarySectionName.DNNmem], in_docker=in_docker)

================== Create docker containers ==================


100%|██████████| 1/1 [00:00<00:00, 43.13it/s]


================== Execute docker containers ==================
=============== Start massively run for GPU train ======================


100%|██████████| 1/1 [00:25<00:00, 25.90s/it]
0it [00:00, ?it/s]

================== Statistics ==================
Run(success/total): 1/1


## Estimate Max GPU Memory by SchedTune

In [9]:
exp.run_estimation(estimators=[SummarySectionName.schedtune], in_docker=True)

================== Create docker containers ==================


100%|██████████| 1/1 [00:00<00:00, 10.39it/s]


================== Execute docker containers ==================
=============== Start massively run for GPU train ======================


0it [00:00, ?it/s]
100%|██████████| 1/1 [00:20<00:00, 20.90s/it]

================== Statistics ==================
Run(success/total): 1/1


In [10]:
exp.statistics()
exp.to_evaluation_result()


100%|██████████| 1/1 [00:00<00:00, 7752.87it/s]


=============== Statistics for Transformer-Exp ==================
train: 1/1
config: 1/1
groundtruth: 1/1
solution: 1/1
schedtune: 0/1
DNNmem: 1/1
LLmem: 0/1


100%|██████████| 1/1 [00:00<00:00, 3554.49it/s]


[{'memory': 1983905792,
  'oom': False,
  'runtime': 1113062815,
  'ground': 3273654272,
  'error': 0.3939782190903267,
  'real_oom': False,
  'correct_estimation': True,
  '2nd verification': {'oom': True, 'error': None},
  'tool': 'DNNmem'},
 {'memory': 3521118208,
  'oom': False,
  'runtime': 21724650676,
  'ground': 3273654272,
  'error': 0.07559256886611147,
  'real_oom': False,
  'correct_estimation': True,
  '2nd verification': {'oom': True, 'error': None},
  'tool': 'Solution'}]